In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Měření Qubitů

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme použít tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Chceš-li získat informace o stavu Qubitu, můžeš ho _změřit_ na [klasický bit](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit). V Qiskitu se měření provádějí ve výpočetní bázi, tj. v jednoqubitové Pauliho-$Z$ bázi. Měření tedy vrátí hodnotu 0 nebo 1 podle překryvu s vlastními stavy Pauliho-$Z$ operátoru $|0\rangle$ a $|1\rangle$:

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## Měření uprostřed Circuit
Měření uprostřed Circuit jsou klíčovou součástí dynamických Circuit. Před verzí `qiskit-ibm-runtime` v0.43.0 byla instrukce `measure` jedinou měřicí instrukcí v Qiskitu. Měření uprostřed Circuit však mají jiné požadavky na ladění než _terminální_ měření (měření probíhající na konci Circuit). Například při ladění měření uprostřed Circuit je třeba brát v úvahu dobu trvání instrukce, protože delší instrukce způsobují větší šum. U terminálních měření dobu trvání instrukce řešit nemusíš, protože po nich již žádné instrukce nenásledují.

Ve verzi `qiskit-ibm-runtime` v0.43.0 byla zavedena instrukce `MidCircuitMeasure`. Jak název napovídá, jde o novou měřicí instrukci optimalizovanou pro měření uprostřed Circuit na IBM&reg; QPU.

> **Note:** Instrukce `MidCircuitMeasure` se mapuje na instrukci `measure_2` uvedenou v `supported_instructions` Backendu. Instrukce `measure_2` však není podporována na všech Backendech. Pomocí `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` najdeš Backendy, které ji podporují. V budoucnu mohou být přidány nové instrukce měření, ale není to zaručeno.

## Přidání měření do Circuit
Existuje několik způsobů, jak přidat měření do Circuit:

### Metoda `QuantumCircuit.measure`
Metodu [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure) použij k měření [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class).

Příklady:

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### Třída `Measure`
Třída [Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure) v Qiskitu změří zadané Qubity.

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

## Next steps

<Admonition type="tip" title="Recommendations">
- [`Measure`](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class
- [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method
- [`measure_active`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active) method
- [`random_circuit`](/docs/api/qiskit/circuit_random#qiskit.circuit.random.random_circuit) method
- [Mid-circuit measurements](/docs/guides/execute-dynamic-circuits#midcircuit) (Available only when using Qiskit Runtime.)
</Admonition>